# AudioLibro Libre on Google Colab (optional)

Convert a book into an audiobook **without installing anything on your computer**, using a computer provided by Google.
Use this notebook only if the local computer is too slow or software cannot be installed on it.

## Limitations: please read before starting
- **The book is uploaded to a Google server** for the duration of the session. This contradicts the "everything runs locally" principle of AudioLibro Libre. If the book is private or sensitive, **do not use Colab**: install the application locally instead.
- A **Google account** (free) and a continuous internet connection are required.
- Free sessions **disconnect** after a period of inactivity or after a few hours, and **everything stored in them is erased** (unless the optional Google Drive cell is used).
- Free Colab usually provides **few CPU cores**, so it is not guaranteed to be faster than an ordinary laptop.
- Users must hold the **rights to the book** they convert.
- This notebook **could not be tested** in the environment where the project was developed; if something fails, please report it in the repository.

## How to use it
Run the cells from top to bottom (the play button on each cell). When the last cell displays the link, open it: it shows the same five-step application.

---

## Español

# AudioLibro Libre en Google Colab (opcional)

Convierte un libro en audiolibro **sin instalar nada en el ordenador local**, usando un ordenador proporcionado por Google.
Úsalo solo si tu ordenador es muy lento o no puedes instalar programas.

## Limitaciones — léelas antes de empezar
- **Tu libro se sube a un servidor de Google** mientras dure la sesión. Esto contradice la idea de "todo en local" de AudioLibro Libre. Si el libro es privado o sensible, **no uses Colab**: instala la aplicación en tu ordenador.
- Necesitas una **cuenta de Google** (gratuita) e internet constante.
- La sesión gratuita **se desconecta** si estás inactivo un rato o tras unas horas, y **se borra todo** lo que hay en ella (a menos que uses la celda opcional de Google Drive).
- El Colab gratuito suele dar **pocos núcleos de CPU**: no está garantizado que vaya más rápido que un portátil normal.
- Recuerda: **debes tener los derechos del libro que conviertes.**
- Este notebook **no se ha podido probar** en el entorno donde se desarrolló el proyecto; si algo falla, avisa en el repositorio.

## Cómo usarlo
Ejecuta las celdas de arriba abajo (botón ▶ de cada una). Cuando la última celda muestre el enlace, ábrelo: verás la misma aplicación de 5 pasos.

In [ ]:
# 1) System programs (ffmpeg, espeak-ng and Tesseract for scanned PDFs) / Programas del sistema
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng tesseract-ocr tesseract-ocr-spa tesseract-ocr-eng > /dev/null
!ffmpeg -version | head -1 && espeak-ng --version | head -1

## 2) Obtain AudioLibro Libre
Two options: **paste the address** of the release ZIP (shown on the repository's *Releases* page) or **leave it empty** and upload the ZIP from your computer when prompted.

## 2) Traer AudioLibro Libre
Dos opciones: **pega la dirección** del ZIP de la versión (la que aparece en la página de *Releases* del repositorio) o **déjala vacía** y sube el ZIP desde tu ordenador cuando te lo pida.

In [ ]:
#@title Release ZIP address (optional) / Dirección del ZIP de la versión (opcional)
RELEASE_URL = ""  #@param {type:"string"}

import glob
import os
import urllib.request
import zipfile

os.chdir("/content")
if RELEASE_URL.strip():
    urllib.request.urlretrieve(RELEASE_URL.strip(), "audiolibro.zip")
else:
    from google.colab import files
    up = files.upload()                      # elige el ZIP audiolibro-libre-X.Y.Z.zip
    os.replace(next(iter(up)), "audiolibro.zip")
with zipfile.ZipFile("audiolibro.zip") as z:
    z.extractall("/content/app_src")
SRC = os.path.dirname(glob.glob("/content/app_src/*/pyproject.toml")[0])
print("Código en:", SRC)

In [ ]:
# 3) Python dependencies (1-3 minutes) / Dependencias de Python (1-3 minutos)
import subprocess

subprocess.run(["pip", "install", "-q", "-e", SRC], check=True)
print("Dependencias instaladas.")

## (Optional) Save projects to Google Drive
Without this, **everything is lost when Colab is closed**. With it, the book and the audio are stored in *your* Drive (again, on Google servers).
Remove the `#` from the two lines below to enable it.

## (Opcional) Guardar tus proyectos en Google Drive
Sin esto, **todo se pierde al cerrar Colab**. Con esto tu libro y el audio se guardan en *tu* Drive (otra vez: en servidores de Google).
Quita el `#` de las dos líneas siguientes para activarlo.

In [ ]:
DATA_DIR = "/content/audiolibro-data"
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/AudioLibroLibre"
os.makedirs(DATA_DIR, exist_ok=True)
print("Datos en:", DATA_DIR)

## (Experimental, untested) Accelerate with a GPU
Use **Runtime > Change runtime type > T4 GPU**, and run this cell **before** starting the application.
A GPU is optional and can only accelerate processing; if anything fails, return to a runtime without a GPU.

## (Experimental, sin probar) Acelerar con GPU
Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** y ejecuta esta celda **antes** de iniciar la aplicación.
La GPU es opcional y solo puede acelerar; si algo falla, vuelve a un entorno sin GPU.

In [ ]:
# Quita el # para usar la GPU (reemplaza onnxruntime por la versión con CUDA)
# !pip uninstall -y -q onnxruntime && pip install -q onnxruntime-gpu

## 4) Start the application

## 4) Iniciar la aplicación

In [ ]:
import subprocess
import sys
import time
import urllib.request

env = dict(os.environ, AUDIOLIBRO_DATA=DATA_DIR, AUDIOLIBRO_NO_BROWSER="1", AUDIOLIBRO_PORT="8000")
server = subprocess.Popen([sys.executable, "-m", "app"], cwd=SRC, env=env,
                          stdout=open("/content/audiolibro.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/system", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/audiolibro.log").read()[-2000:]); raise SystemExit("La aplicación no arrancó (ver registro arriba).")

from google.colab import output

output.serve_kernel_port_as_window(8000)   # abre la aplicación en una pestaña nueva
print("Si no se abrió, pulsa el enlace que aparece arriba. NO cierres esta pestaña de Colab mientras trabajas.")

## 5) When finished
In step 5 of the application, select **"Download ZIP"** to take the audiobook to your computer. Then stop the previous cell and close Colab; remember that **everything else is erased** (unless Drive was used).

## 5) Cuando termines
En el paso 5 de la aplicación pulsa **«Descargar ZIP»**: así te llevas el audiolibro a tu ordenador. Después puedes parar la celda anterior (⏹) y cerrar Colab; recuerda que **todo lo demás se borra** (salvo que hayas usado Drive).